In [45]:
import sys
import os
project_root = os.path.abspath("..")
sys.path.append(project_root)

import torch
import yaml
import re
import glob
from pathlib import Path

from main import LVSMLauncherConfig
from src.models.lvsm_decoder_only import LVSMDecoderOnlyModel
from src.configs.lvsm_decoder_only_config import (
    LVSMDecoderOnlyModelConfig,
    RayEncodingType,
    PosEncType,
)
from src.prope.utils.transformer import (
    TransformerEncoderConfig,
    TransformerEncoderLayerConfig,
)

device = "cuda" if torch.cuda.is_available() else "cpu"

# run_dir = "../results/nvs/PX/release-1gpus-b8-s1-80k-CAMRAY-PROPE/" #PX
# run_dir = "../results/nvs/VAE/release-1gpus-b8-s1-80k-CAMRAY-PROPE/" #PX
run_dir = "../results/nvs/release-1gpus-b8-s1-80k-CAMRAY-PROPE/" #VAE

scale_factor = 2
patch_size = 8



def get_config(dir):
    # -------------------------------
    # Load & clean YAML
    # -------------------------------    
    raw = Path(f"{dir}/config.yaml").read_text()
    clean = re.sub(r"!!python/[^ \n]+", "", raw)   # strip python tags

    config_dict = yaml.safe_load(clean)


    # -------------------------------
    # Rebuild the launcher config
    # -------------------------------
    cfg = LVSMLauncherConfig()

    for k, v in config_dict.items():
        if k != "model_config":   # we'll rebuild this separately
            setattr(cfg, k, v)


    # -------------------------------
    # Rebuild the nested model config
    # -------------------------------
    m = config_dict["model_config"]

    # Enums were dumped as single-item lists
    ray_encoding = RayEncodingType(m["ray_encoding"][0])
    pos_enc      = PosEncType(m["pos_enc"][0])

    # Encoder layer
    layer_cfg = m["encoder"]["layer"]

    encoder_layer = TransformerEncoderLayerConfig(
        d_model=layer_cfg["d_model"],
        nhead=layer_cfg["nhead"],
        dim_feedforward=layer_cfg["dim_feedforward"],
        dropout=layer_cfg["dropout"],
        activation=torch.nn.functional.relu,  # original activation
        batch_first=layer_cfg["batch_first"],
        bias=layer_cfg["bias"],
        layer_norm_eps=layer_cfg["layer_norm_eps"],
        modulation_activation=layer_cfg["modulation_activation"],
        norm_first=layer_cfg["norm_first"],
        norm_type=layer_cfg["norm_type"],
        elementwise_affine=layer_cfg["elementwise_affine"],
        qk_norm=layer_cfg["qk_norm"],
    )

    encoder = TransformerEncoderConfig(
        layer=encoder_layer,
        num_layers=m["encoder"]["num_layers"],
        input_norm=m["encoder"]["input_norm"],
        output_norm=m["encoder"]["output_norm"],
        checkpointing=m["encoder"]["checkpointing"],
    )

    # Full model config
    model_cfg = LVSMDecoderOnlyModelConfig(
        ref_views=m["ref_views"],
        tar_views=m["tar_views"],
        encoder=encoder,
        img_shape=tuple([m["img_shape"][0]*scale_factor, m["img_shape"][1]*scale_factor,m["img_shape"][2]]),
        cam_shape=tuple([m["cam_shape"][0]*scale_factor, m["cam_shape"][1]*scale_factor,m["cam_shape"][2]]),        
        patch_size=patch_size,
        # patch_size=m["patch_size"]*latent_patch_scale,
        ray_encoding=ray_encoding,
        pos_enc=pos_enc,
    )

    # The launcher should store this model_config
    cfg.model_config = model_cfg
    return cfg

def load_checkpoint(model, dir):
    def get_latest_checkpoint(ckpt_dir):
        ckpts = sorted(glob.glob(os.path.join(ckpt_dir, "*.pt")))
        return ckpts[-1] if ckpts else None

    ckpt_path = get_latest_checkpoint(f"{dir}/ckpts")
    print("Loading checkpoint:", ckpt_path)

    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=True)
    state_dict = {k.replace("_orig_mod.", ""): v for k, v in ckpt["model"].items()}
    missing, unexpected = model.load_state_dict(state_dict, strict=False)

    print("Missing keys:", missing)
    print("Unexpected keys:", unexpected)
    return model

cfg = get_config(run_dir)


model = LVSMDecoderOnlyModel(cfg.model_config).to(device)
# model = load_checkpoint(model, run_dir)
model.eval()

print("Model space:", cfg.model_space)

Model space: VAE


In [46]:
import glob

from src.data.dataset import TrainDataset
from src.data.latent_dataset import TrainLatentDataset
from main import LVSMLauncher, LVSMLauncherConfig


def load_single_batch(cfg: LVSMLauncherConfig):
    """
    Loads ONE batch from the appropriate dataset based on cfg.model_space.
    Returns ref_imgs, tar_imgs, ref_cams, tar_cams, ref_paths, tar_paths.
    """

    launcher = LVSMLauncher(cfg)
    
    if cfg.model_space == "PX":
        scenes = sorted(glob.glob("../data/data_processed/realestate10k/train/*"))
        dataset = TrainDataset(
            scenes,
            patch_size=cfg.dataset_patch_size,
            zoom_factor=cfg.train_zoom_factor,
            random_zoom=cfg.random_zoom,
            supervise_views=cfg.dataset_supervise_views,
            upscale_factor=scale_factor
        )
    else:
        scenes = sorted(glob.glob("../data/data_processed/realestate10k_latent/train/*"))
        dataset = TrainLatentDataset(
            scenes,
            supervise_views=cfg.dataset_supervise_views,            
            upscale_factor=scale_factor
        )    

    data = dataset[0] # just one item
    data["image"] = data["image"].unsqueeze(0)
    data["K"] = data["K"].unsqueeze(0)
    data["camtoworld"] = data["camtoworld"].unsqueeze(0)
    data["image_path"] = [data["image_path"]]  # wrap in list
    
    input_views = data["K"].shape[1] - cfg.dataset_supervise_views
    
    processed = launcher.preprocess(data, input_views=input_views)

    ref_imgs = processed["ref_imgs"]
    tar_imgs = processed["tar_imgs"]
    ref_cams = processed["ref_cams"]
    tar_cams = processed["tar_cams"]
    ref_paths = processed["ref_paths"]
    tar_paths = processed["tar_paths"]

    return ref_imgs, tar_imgs, ref_cams, tar_cams, ref_paths, tar_paths


In [47]:
from main import LVSMLauncherConfig



ref_imgs, tar_imgs, ref_cams, tar_cams, ref_paths, tar_paths = load_single_batch(cfg)

print("ref_imgs:", ref_imgs.shape)
print("tar_imgs:", tar_imgs.shape)
print("latent?" if cfg.model_space == "VAE" else "pixels")

Wrote config to results/nvs/release-1gpus-b8-s1-80k-CAMRAY-PROPE/config.yaml
training on latent dataset
ref_imgs: torch.Size([1, 2, 64, 64, 16])
tar_imgs: torch.Size([1, 1, 64, 64, 16])
latent?


In [48]:
launcher = LVSMLauncher(cfg)

with torch.inference_mode():
    out = model(ref_imgs.to(device), ref_cams, tar_cams)

if cfg.model_space == "PX":
    out = torch.sigmoid(out)
else:
    img = launcher.decode_tensors(out[0])

out

Wrote config to results/nvs/release-1gpus-b8-s1-80k-CAMRAY-PROPE/config.yaml


tensor([[[[[ 0.3054, -0.4920, -0.1444,  ...,  0.2338, -0.6922,  0.3083],
           [-0.3974,  0.5348, -0.5229,  ...,  0.2762, -0.5761,  0.2894],
           [-0.0948, -0.3682, -0.9645,  ..., -0.3862, -0.3145,  0.0250],
           ...,
           [ 0.0506,  0.1901, -0.1060,  ...,  0.3578,  0.3679,  0.9483],
           [-0.1004, -0.8174,  1.2737,  ...,  1.0429, -0.4367,  0.5511],
           [-0.1171,  0.0874,  0.5259,  ...,  0.3816,  0.8597, -1.8352]],

          [[-0.2887, -0.3162, -0.1005,  ..., -0.5006, -1.6773,  0.8413],
           [ 0.4260,  0.2261, -0.0263,  ..., -0.2350, -0.5395, -0.4326],
           [ 0.2790,  0.3169, -0.8633,  ...,  0.2056, -0.3204, -0.1829],
           ...,
           [ 0.0998, -0.3102,  0.6073,  ..., -1.3367, -0.2353, -0.3808],
           [-0.6735,  0.6886, -0.3300,  ..., -0.5316, -0.8562,  0.6051],
           [-1.3251, -1.0709, -0.1620,  ...,  0.3201,  0.3117,  0.1065]],

          [[ 0.7525,  0.5518,  1.2971,  ...,  0.1073,  0.1215, -0.2957],
           [ 0.

In [49]:
import time
import torch
import torch.nn.functional as F

def tensor_bytes(x: torch.Tensor) -> int:
    return x.numel() * x.element_size()

def encode_tensors(img: torch.Tensor, vae, device) -> torch.Tensor:
    """
    Accepts:
        [H, W, 3]
        [V, H, W, 3]
        [B, V, H, W, 3]

    Returns:
        Same leading dims, but encoded latents:
        [.., H_down, W_down, C_latent]
    """

    img = img.to(device).float()

    # Original leading dims: (), (V), or (B, V)
    orig_shape = img.shape[:-3]
    H, W, C = img.shape[-3:]
    assert C == 3, f"Expected RGB input but got {C} channels"

    # Flatten leading dims
    img = img.reshape(-1, H, W, C)        # [N, H, W, 3]

    # Convert to NCHW
    img = img.permute(0, 3, 1, 2)         # [N, 3, H, W]

    # Scale to [-1, 1] just like training
    img = img * 2 - 1

    with torch.no_grad():
        posterior = vae.encode(img)
        latents = posterior.latent_dist.mean   # [N, C_lat, H_down, W_down]

    # Back to NHWC
    latents = latents.permute(0, 2, 3, 1)   # [N, H_down, W_down, C_lat]


    # Restore original leading dims
    latents = latents.reshape(*orig_shape,
                              latents.shape[1],
                              latents.shape[2],
                              latents.shape[3])

    return latents


def scale_spatial(img: torch.Tensor, factor: int) -> torch.Tensor:
    """
    Inputs:
        img: [B, V, H, W, C]
    Returns:
        [B, V, H*factor, W*factor, C]

    Uses bilinear interpolation. Assumes img is on any device.
    """

    B, V, H, W, C = img.shape
    img_nchw = img.permute(0, 1, 4, 2, 3).reshape(B*V, C, H, W)

    img_up = F.interpolate(
        img_nchw,
        size=(H * factor, W * factor),
        mode="bilinear",
        align_corners=False
    )
    img_up = img_up.reshape(B, V, C, H*factor, W*factor).permute(0, 1, 3, 4, 2)

    return img_up


def run_inference_with_metrics(model, launcher, ref_imgs, tar_cams, cfg, device="cuda", scale_factor=1):    

    # ------------------------------------------
    # Model parameter stats
    # ------------------------------------------
    model_params = sum(p.numel() for p in model.parameters())

    # ------------------------------------------
    # Input size (bytes)
    # ------------------------------------------
    input_tensors = [ref_imgs]
    input_bytes = sum(tensor_bytes(t) for t in input_tensors if isinstance(t, torch.Tensor))
    


    # These final numbers we will fill:
    forward_memory_mb = 0.0
    encode_memory_mb = 0.0
    decode_memory_mb = 0.0
    peak_memory_mb = 0.0
    encode_peak = 0.0
    encode_time_ms = 0.0

    t_enc1 = time.time()

    


    if cfg.model_space == "VAE":
        decoded = launcher.decode_tensors(ref_imgs)                    
        if scale_factor > 1:
            decoded = scale_spatial(decoded, scale_factor)
        t_enc1 = time.time()
        encode_tensors(decoded, launcher.vae32, device)
        encode_time_ms = (time.time() - t_enc1)* 1000
        if torch.cuda.is_available():
            encode_memory_mb = torch.cuda.max_memory_allocated(device) / (1024**2)
        encode_peak = torch.cuda.max_memory_allocated(device)

    # ------------------------------------------
    # Forward pass timing + memory
    # ------------------------------------------
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats(device)

    torch.cuda.synchronize()
    t_fwd0 = time.time()

    with torch.inference_mode():
        out = model(ref_imgs.to(device), ref_cams, tar_cams)

    torch.cuda.synchronize()
    t_fwd1 = time.time()

    forward_time_ms = (t_fwd1 - t_fwd0) * 1000

    # Memory AFTER forward pass
    if torch.cuda.is_available():
        forward_memory_mb = torch.cuda.max_memory_allocated(device) / (1024**2)

    # Keep the forward peak BEFORE decoding wipes it out
    forward_peak = torch.cuda.max_memory_allocated(device)

    # ------------------------------------------
    # Decoding (only for VAE)
    # ------------------------------------------
    if cfg.model_space == "VAE":

        # Reset for decode-only measurement
        torch.cuda.reset_peak_memory_stats(device)
        torch.cuda.synchronize()
        t_dec0 = time.time()

        decoded = launcher.decode_tensors(out[0])

        torch.cuda.synchronize()
        t_dec1 = time.time()
        decode_time_ms = (t_dec1 - t_dec0) * 1000

        decode_memory_mb = torch.cuda.max_memory_allocated(device) / (1024**2)

    else:
        decoded = torch.sigmoid(out)
        decode_time_ms = 0.0
        decode_memory_mb = 0.0

    # ------------------------------------------
    # Total peak memory (full process)
    # ------------------------------------------
    peak_memory_mb = max(encode_peak, forward_peak, torch.cuda.max_memory_allocated(device)) / (1024**2)

    # ------------------------------------------
    # Total wall time
    # ------------------------------------------
    total_time_ms = forward_time_ms + decode_time_ms + encode_time_ms

    # ------------------------------------------
    # Return metrics
    # ------------------------------------------
    return {
        "model_params": model_params,
        "input_bytes": input_bytes,
        "input_MB": input_bytes / (1024**2),
        "encode_time_ms": encode_time_ms,
        "forward_time_ms": forward_time_ms,
        "decode_time_ms": decode_time_ms,
        "total_time_ms": total_time_ms,
        "encode_memory_mb": encode_memory_mb,
        "forward_memory_mb": forward_memory_mb,
        "decode_memory_mb": decode_memory_mb,
        "peak_memory_mb": peak_memory_mb,
        "output_shape": tuple(out.shape),
        "decoded_shape": tuple(decoded.shape),
    }


In [50]:
launcher = LVSMLauncher(cfg)

metrics = run_inference_with_metrics(
    model=model,
    launcher=launcher,
    ref_imgs=ref_imgs,
    tar_cams=tar_cams,
    cfg=cfg,
    scale_factor=scale_factor
)

import json
print(json.dumps(metrics, indent=4))

import gc
import torch

gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()



Wrote config to results/nvs/release-1gpus-b8-s1-80k-CAMRAY-PROPE/config.yaml
{
    "model_params": 17899392,
    "input_bytes": 524288,
    "input_MB": 0.5,
    "encode_time_ms": 1061.1310005187988,
    "forward_time_ms": 40.18449783325195,
    "decode_time_ms": 451.9462585449219,
    "total_time_ms": 1553.2617568969727,
    "encode_memory_mb": 5580.72509765625,
    "forward_memory_mb": 443.6962890625,
    "decode_memory_mb": 1204.97509765625,
    "peak_memory_mb": 5580.72509765625,
    "output_shape": [
        1,
        1,
        64,
        64,
        16
    ],
    "decoded_shape": [
        1,
        512,
        512,
        3
    ]
}
